# End-to-end SkySurvey Warp sample

This notebook runs the complete schema-6 WarpTemplate workflow in nine ordered stages: configuration, population validation, cache construction, survey-realization planning, simulation setup, sequential simulation execution, representative selection, magnitude comparison, and provenance inspection.

The default configuration uses the complete non-overlapping, rate-ready CC leaf set: `SN IIP`, `SN IIb`, `SN IIn`, `SN Ib`, `SN Ic`, `SN Ic-BL`, `SLSN-I`, and `SLSN-II`. For classifier sampling, the two raw SLSN leaves share one merged `SLSN` quota, so each of the seven final classes receives approximately one seventh of the requested sample; the SLSN quota is then divided evenly between `SLSN-I` and `SLSN-II`. Aggregate aliases are not simulated directly. `SN Ibn` has Warp coefficients but is not included because its rate is deliberately marked `missing_direct_rate` in the literature configuration. The sample is divided across many 360-day cadence realizations whose starts are spread across each full survey archive; neighboring windows may overlap to provide dense cadence diversity. Event peaks stay at least 60 observer-frame days from the left edge and 220 days from the right edge, which covers the configured rest-frame phase range even at `zmax`. Combined mode still aligns independent ZTF and LSST windows onto common simulation dates and is therefore not a forecast of coordinated observations.

## 1. Central configuration

Edit only the clearly marked **USER SETTINGS** block at the top of the next cell, then select **Run All**. Everything below **INTERNAL SETUP** is derived automatically and normally does not need to be changed.

| Section | Important settings | Meaning |
|---|---|---|
| Survey | `survey_name` | `"ztf"`, `"lsst"`, or `"combined"` |
| Population | `sample_size`, `seed`, `active_fitclasses`, `zmax` | Number, reproducibility, classes, and maximum redshift |
| Time window | `duration_days`, `observation_tstart`, peak margins | Cadence duration and allowed peak-time region |
| Survey diversity | `survey_realization_count`, `require_non_overlapping_realizations` | Number of archive-wide windows and optional strict independence |
| Plot selection | `minimum_plot_snr`, `maximum_example_detections` | Detection threshold and strict upper example limit |
| Cadence details | `ztf_filters`, `lsst_filters`, `ztf_clean_only`, `survey_nside` | Loaded bands and spatial proposal resolution |
| Scaling | `batch_size`, `max_sources_per_batch`, `progress_every_batches`, `keep_warpcoeffs_in_memory` | Memory bounds, coefficient reuse, and live progress |
| Analysis safety | `analysis_objects_per_fitclass` | Limits only the in-memory plotting subset, not the persisted simulation |
| Advanced | `simulation_phase_range`, `include_noise`, `custom_*` | Light-curve scope, noise, and optional paths/run name |

The configured 360-day cadence windows are placed evenly across every available survey archive and may overlap. They are loaded and simulated sequentially, so only one realization is resident at a time. The total `sample_size` is divided across them; it is not repeated for every realization. Event peaks are drawn only inside a protected interior window. A value of `None` for a custom source-start list lets the planner choose evenly distributed windows automatically. Set `require_non_overlapping_realizations=True` only when statistically independent cadence blocks are specifically required and the requested number fits.

For stress testing, first record the 512-target baseline, then increase `sample_size` to `10_000`, `100_000`, and finally `1_000_000`. Keep `resume=True`, change only one scaling control at a time, and compare the final timing tables. Do not set `analysis_objects_per_fitclass=None` at million scale unless the complete observation table is known to fit in memory.

In [ ]:
# Configure the complete notebook from one human-editable settings block
# ==============================================================================
# USER SETTINGS — EDIT VALUES IN THIS BLOCK, THEN SELECT "RUN ALL"
# ==============================================================================

# Survey and sample size
survey_name = "combined"  # Choose "ztf", "lsst", or "combined".
sample_size = 100_000  # Stress ladder: 10_000 -> 100_000 -> 1_000_000.
seed = 20260720  # Change this integer to draw a different reproducible sample.

# Transient classes and redshift range
active_fitclasses = [
    "SN IIP",
    "SN IIb",
    "SN IIn",
    "SN Ib",
    "SN Ic",
    "SN Ic-BL",
    "SLSN-I",
    "SLSN-II",
]
zmax = 0.06  # Redshifts are drawn uniformly from 0 to this value.

# Observing window and survey diversity
observation_tstart = 60000.0
duration_days = 360.0
pre_peak_margin_days = 53.0  # Covers phase -50 d through z=0.06.
post_peak_margin_days = 212.0  # Covers phase +200 d through z=0.06.
survey_realization_count = 36  # Total sample is divided across these windows.
require_non_overlapping_realizations = False  # Sliding windows may overlap.
custom_ztf_source_mjd_starts = None  # Or one MJD per realization.
custom_lsst_source_mjd_starts = None  # Or one MJD per realization.

# Plot detection and example-selection settings
minimum_plot_snr = 1.0
maximum_example_detections = 100  # Selection uses the largest count below this.
comparison_redshift = 0.03  # Used only to break equal-detection-count ties.

# Survey bands and spatial proposal resolution
ztf_filters = ["g", "r", "i"]
lsst_filters = ["u", "g", "r", "i", "z", "y"]
ztf_clean_only = True
survey_nside = 64

# Scaling controls: increase gradually and watch the performance summary.
batch_size = 128  # Higher is faster but uses more peak RAM.
max_sources_per_batch = 128  # Upper bound for distinct cached source splines.
progress_every_batches = 25  # Use None to disable live batch progress.
keep_warpcoeffs_in_memory = True  # Reuse them across all realizations.
analysis_objects_per_fitclass = 1000  # None loads every result into RAM.

# Advanced simulation controls
simulation_phase_range = (-50.0, 200.0)  # Rest-frame days retained.
include_noise = True
show_stage_timings = True

# Optional locations: None keeps the automatic path or run name.
custom_run_name = None
custom_source_cache_dir = None
custom_output_dir = None

# ==============================================================================
# INTERNAL SETUP — VALUES BELOW ARE DERIVED AUTOMATICALLY
# ==============================================================================
from hashlib import blake2b
from pathlib import Path
import gc
import importlib
import json
import os
import resource
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

stage_metrics = []

def peak_rss_gib():
    """Return this kernel's peak resident memory in GiB on Linux."""

    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2

def start_stage(stage):
    """Start one visible notebook-stage timer."""

    if show_stage_timings:
        print(f"[Stage start] {stage}")
    return time.perf_counter(), peak_rss_gib()

def finish_stage(stage, started, **details):
    """Record elapsed time, peak memory, and optional stage-size details."""

    started_at, peak_before_gib = started
    elapsed_seconds = time.perf_counter() - started_at
    peak_after_gib = peak_rss_gib()
    detail_parts = []
    for name, value in details.items():
        if isinstance(value, (int, np.integer)):
            rendered = f"{int(value):,}"
        elif isinstance(value, (float, np.floating)):
            rendered = f"{float(value):,.2f}"
        else:
            rendered = str(value)
        detail_parts.append(f"{name}={rendered}")
    metric = {
        "stage": stage,
        "seconds": elapsed_seconds,
        "peak_rss_gib": peak_after_gib,
        "peak_increase_gib": max(peak_after_gib - peak_before_gib, 0.0),
        "details": ", ".join(detail_parts),
    }
    stage_metrics.append(metric)
    if show_stage_timings:
        print(
            f"[Stage done]  {stage}: {elapsed_seconds:,.2f} s | "
            f"peak RSS {metric['peak_rss_gib']:.2f} GiB"
        )
    return metric

configuration_started = start_stage("1. Configuration")

# Locate the project root independently of the directory from which Jupyter starts.
repo_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "warpTemplate").exists()
    and (path / "data" / "warpcoeff_v3").exists()
)
# Import SkySurvey from the kernel while the workspace reference copy is hidden.
original_sys_path = list(sys.path)
kernel_sys_path = []
for entry in original_sys_path:
    try:
        resolved_entry = Path(entry or Path.cwd()).resolve()
    except (OSError, RuntimeError):
        resolved_entry = None
    if resolved_entry != repo_root:
        kernel_sys_path.append(entry)
for module_name in [name for name in sys.modules if name == "skysurvey" or name.startswith("skysurvey.")]:
    sys.modules.pop(module_name, None)
try:
    sys.path[:] = kernel_sys_path
    skysurvey = importlib.import_module("skysurvey")
finally:
    sys.path[:] = original_sys_path
skysurvey_path = Path(skysurvey.__file__).resolve()
if repo_root in skysurvey_path.parents or not hasattr(skysurvey, "DataSet"):
    raise RuntimeError(
        "The active kernel must provide the installed astronomical skysurvey package; "
        f"resolved {skysurvey_path}"
    )

# Only WarpTemplate itself is intentionally imported from this checkout.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from warpTemplate.batch_simulation import (
    SIMULATION_SCHEMA_VERSION,
    allocate_group_balanced_counts,
)

# Normalize user input and derive all dependent time and batching values.
survey_name = str(survey_name).strip().lower()
sample_size = int(sample_size)
seed = int(seed)
survey_realization_count = int(survey_realization_count)
require_non_overlapping_realizations = bool(
    require_non_overlapping_realizations
)
duration_days = float(duration_days)
observation_tstart = float(observation_tstart)
pre_peak_margin_days = float(pre_peak_margin_days)
post_peak_margin_days = float(post_peak_margin_days)
zmax = float(zmax)
comparison_redshift = float(comparison_redshift)
minimum_plot_snr = float(minimum_plot_snr)
maximum_example_detections = int(maximum_example_detections)
include_noise = bool(include_noise)
keep_warpcoeffs_in_memory = bool(keep_warpcoeffs_in_memory)
survey_nside = int(survey_nside)
ztf_filters = [str(name).strip() for name in ztf_filters]
lsst_filters = [str(name).strip() for name in lsst_filters]
batch_size = min(sample_size, int(batch_size))
max_sources_per_batch = int(max_sources_per_batch)
progress_every_batches = (
    None
    if progress_every_batches is None
    else int(progress_every_batches)
)
analysis_objects_per_fitclass = (
    None
    if analysis_objects_per_fitclass is None
    else int(analysis_objects_per_fitclass)
)
simulation_phase_range = tuple(map(float, simulation_phase_range))
active_fitclasses = list(
    dict.fromkeys(
        str(name).strip()
        for name in active_fitclasses
        if str(name).strip()
    )
)
# Balance the classifier taxonomy first; raw SLSN leaves divide one parent quota.
slsn_fitclasses = [
    name for name in ("SLSN-I", "SLSN-II") if name in active_fitclasses
]
balanced_class_groups = {}
if slsn_fitclasses:
    balanced_class_groups["SLSN"] = slsn_fitclasses
balanced_class_groups.update(
    {name: [name] for name in active_fitclasses if name not in slsn_fitclasses}
)
observation_tstop = observation_tstart + duration_days
peak_tstart = observation_tstart + pre_peak_margin_days
peak_tstop = observation_tstop - post_peak_margin_days

# Reject inconsistent windows and undersized samples before any expensive data access.
if survey_name not in {"ztf", "lsst", "combined"}:
    raise ValueError('survey_name must be "ztf", "lsst", or "combined"')
if not active_fitclasses:
    raise ValueError("active_fitclasses must contain at least one class")
if duration_days <= 0.0:
    raise ValueError("duration_days must be positive")
if survey_realization_count <= 0:
    raise ValueError("survey_realization_count must be positive")
if pre_peak_margin_days < 0.0 or post_peak_margin_days < 0.0:
    raise ValueError("pre- and post-peak margins must be non-negative")
if sample_size < survey_realization_count * len(balanced_class_groups):
    raise ValueError(
        "sample_size must provide at least one object per final class and realization"
    )
if zmax < 0.0:
    raise ValueError("zmax must be non-negative")
if minimum_plot_snr < 0.0:
    raise ValueError("minimum_plot_snr must be non-negative")
if maximum_example_detections <= 1:
    raise ValueError("maximum_example_detections must be greater than one")
if batch_size <= 0:
    raise ValueError("batch_size must be positive")
if max_sources_per_batch <= 0:
    raise ValueError("max_sources_per_batch must be positive")
if survey_nside <= 0:
    raise ValueError("survey_nside must be positive")
if survey_name in {"ztf", "combined"} and not ztf_filters:
    raise ValueError("ztf_filters must not be empty for this survey")
if survey_name in {"lsst", "combined"} and not lsst_filters:
    raise ValueError("lsst_filters must not be empty for this survey")
if progress_every_batches is not None and progress_every_batches <= 0:
    raise ValueError("progress_every_batches must be positive or None")
if (
    analysis_objects_per_fitclass is not None
    and analysis_objects_per_fitclass <= 0
):
    raise ValueError("analysis_objects_per_fitclass must be positive or None")
if len(simulation_phase_range) != 2:
    raise ValueError("simulation_phase_range must contain two values")
if simulation_phase_range[0] >= simulation_phase_range[1]:
    raise ValueError("simulation_phase_range must be strictly increasing")
if peak_tstop <= peak_tstart:
    raise ValueError(
        "The observing window must exceed the requested pre- and post-peak margins."
    )
# Report whether edge margins cover the full rest-frame phase range at zmax.
required_pre_peak_days = max(0.0, -simulation_phase_range[0] * (1.0 + zmax))
required_post_peak_days = max(0.0, simulation_phase_range[1] * (1.0 + zmax))
if (
    pre_peak_margin_days < required_pre_peak_days
    or post_peak_margin_days < required_post_peak_days
):
    print(
        "WARNING: edge margins do not cover simulation_phase_range for every "
        f"target at zmax; use at least {required_pre_peak_days:.1f} pre-peak "
        f"and {required_post_peak_days:.1f} post-peak days for full coverage."
    )

# Resolve cache and output locations once so every later stage shares the same paths.
warpcoeffs_dir = repo_root / "data" / "warpcoeff_v3"
source_cache_dir = (
    Path(custom_source_cache_dir).expanduser()
    if custom_source_cache_dir is not None
    else repo_root / "data" / "warp_source_cache_v1"
)
output_dir = (
    Path(custom_output_dir).expanduser()
    if custom_output_dir is not None
    else repo_root / "training_samples"
)

# Keep cadence-loading choices explicit because they define the simulated survey domain.
survey_options = {
    "time_mode": "relative",
    "ztf_filters": list(ztf_filters),
    "lsst_filters": list(lsst_filters),
    "ztf_clean_only": bool(ztf_clean_only),
    "nside": int(survey_nside),
}
survey_geometry_contract = "warp_canonical_periodic_radec_v2"
configuration = {
    "simulation_schema": SIMULATION_SCHEMA_VERSION,
    "survey_geometry_contract": survey_geometry_contract,
    "survey_name": survey_name,
    "sample_size": sample_size,
    "duration_days": duration_days,
    "survey_realization_count": survey_realization_count,
    "require_non_overlapping_realizations": bool(
        require_non_overlapping_realizations
    ),
    "custom_ztf_source_mjd_starts": custom_ztf_source_mjd_starts,
    "custom_lsst_source_mjd_starts": custom_lsst_source_mjd_starts,
    "seed": seed,
    "active_fitclasses": active_fitclasses,
    "class_sampling": "balanced_final_classes",
    "balanced_class_groups": balanced_class_groups,
    "observation_tstart": observation_tstart,
    "observation_tstop": observation_tstop,
    "peak_tstart": peak_tstart,
    "peak_tstop": peak_tstop,
    "pre_peak_margin_days": pre_peak_margin_days,
    "post_peak_margin_days": post_peak_margin_days,
    "zmax": zmax,
    "comparison_redshift": comparison_redshift,
    "minimum_plot_snr": minimum_plot_snr,
    "batch_size": batch_size,
    "max_sources_per_batch": max_sources_per_batch,
    "simulation_phase_range": simulation_phase_range,
    "include_noise": include_noise,
    "survey_options": survey_options,
}
# Hash persisted simulation controls so resumable outputs cannot mix incompatible runs.
configuration_digest = blake2b(
    json.dumps(configuration, sort_keys=True).encode(),
    digest_size=6,
).hexdigest()
automatic_run_name = (
    f"warp_sample_{survey_name}_schema{SIMULATION_SCHEMA_VERSION}_"
    f"{configuration_digest}"
)
run_name = str(custom_run_name) if custom_run_name else automatic_run_name

# Divide the requested total once across realizations, then estimate all batches.
realization_size, realization_remainder = divmod(
    sample_size, survey_realization_count
)
realization_sizes = [
    realization_size + (index < realization_remainder)
    for index in range(survey_realization_count)
]
realization_class_counts = allocate_group_balanced_counts(
    sample_size, balanced_class_groups, shard_sizes=realization_sizes
)
minimum_expected_batches = sum(
    (count + batch_size - 1) // batch_size
    for allocation in realization_class_counts
    for count in allocation.values()
    if count
)
aggregate_raw_counts = {
    fitclass: sum(allocation[fitclass] for allocation in realization_class_counts)
    for fitclass in active_fitclasses
}
aggregate_final_counts = {
    parent: sum(aggregate_raw_counts[member] for member in members)
    for parent, members in balanced_class_groups.items()
}
if max(aggregate_final_counts.values()) - min(aggregate_final_counts.values()) > 1:
    raise RuntimeError("Final classifier classes are not balanced")

# Echo the run settings plus the in-memory example-selection ceiling.
display(
    pd.Series(
        {
            **configuration,
            "maximum_example_detections": maximum_example_detections,
            "progress_every_batches": progress_every_batches,
            "keep_warpcoeffs_in_memory": keep_warpcoeffs_in_memory,
            "analysis_objects_per_fitclass": analysis_objects_per_fitclass,
            "minimum_expected_batches": minimum_expected_batches,
            "run_name": run_name,
        },
        name="value",
    )
)
print(f"Cache:  {source_cache_dir}")
print(f"Output: {output_dir / run_name}")
display(pd.DataFrame({"raw_count": aggregate_raw_counts}))
display(pd.Series(aggregate_final_counts, name="final_class_count"))
if sample_size >= 100_000:
    print(
        f"STRESS RUN: {sample_size:,} targets in at least "
        f"{minimum_expected_batches:,} fitclass-local batches."
    )
finish_stage(
    "1. Configuration",
    configuration_started,
    targets=sample_size,
    minimum_batches=minimum_expected_batches,
)

## 2. Validate the Warp population

Every selected raw class must be present in the coefficient library and fully described by the rate and absolute-magnitude configuration. Sampling is balanced over the seven classifier classes, not the eight raw fitclasses: merged SLSN receives one seventh overall and that quota is split evenly between `SLSN-I` and `SLSN-II`. This is a comparison population, not a rate-representative one.

In [ ]:
# Load and validate WarpTemplate rates, magnitude priors, and selected classes
stage_started = start_stage("2. Validate population")
from warpTemplate import (
    SurveyFactory,
    WarpSampleSpec,
    WarpSimulationRunner,
    WarpSourceCache,
    discover_warp_fitclasses,
    load_warp_rate_config,
    rate_config_to_dataframe,
    validate_active_fitclasses,
    validate_magabs_config,
    validate_rate_config,
)

# Load the scientific population metadata and verify every requested class up front.
rate_config = load_warp_rate_config()
available_fitclasses = discover_warp_fitclasses(warpcoeffs_dir)
validate_rate_config(rate_config, available_fitclasses=available_fitclasses)
validate_magabs_config(rate_config, available_fitclasses=available_fitclasses)
validate_active_fitclasses(active_fitclasses, rate_config)

# Present only the rate and magnitude-prior fields relevant to this simulation.
rate_table = rate_config_to_dataframe(rate_config)
display(
    rate_table.loc[
        active_fitclasses,
        [
            "rate_gpc3_yr",
            "status",
            "magabs_distribution",
            "magabs_source_band",
        ],
    ]
)
finish_stage(
    "2. Validate population",
    stage_started,
    fitclasses=len(active_fitclasses),
)

## 3. Build the lossless Warp source cache

One HDF5 partition is built per active class. Valid partitions matching the coefficient library are reused, and the runner validates them again before simulation.

The cache contains one color-neutral spectral grid per stored Warp entry, not one grid per simulated transient or color draw. Event models share these cached sources. The earlier several-gigabyte estimate was too conservative for this v1 representation: the complete eight-class set is expected to occupy hundreds of megabytes, with the exact size determined by source-grid dimensions and HDF5 compression.

At runtime each source advertises at least 2000--30000 Angstrom coverage so redshifted edge bands such as LSST-u can be integrated. Flux outside the native template grid is explicitly set to zero; this conservative truncation avoids inventing an uncontrolled spectral extrapolation while preserving every cached native value exactly.

In [ ]:
# Build or validate every source-cache partition required by this simulation
stage_started = start_stage("3. Build or validate source cache")
import h5py

# Reuse valid class partitions and rebuild only cache entries that are missing or stale.
source_cache = WarpSourceCache(
    warpcoeffs_dir,
    source_cache_dir,
)
cache_partitions = source_cache.build(active_fitclasses)
cache_table = pd.DataFrame.from_dict(cache_partitions, orient="index")
cache_table.index.name = "fitclass"
if not cache_table["valid"].all():
    raise RuntimeError("At least one required cache partition is invalid.")

def cache_partition_sizes(path):
    """Measure stored and uncompressed array sizes for one cache partition."""

    path = Path(path)
    raw_bytes = 0
    with h5py.File(path, "r") as handle:
        for group in handle["entries"].values():
            raw_bytes += sum(
                group[name].size * group[name].dtype.itemsize
                for name in ("phase", "wave", "flux")
            )
    return pd.Series(
        {
            "stored_mib": path.stat().st_size / 1024**2,
            "raw_array_mib": raw_bytes / 1024**2,
        }
    )

# Compare compressed on-disk size with the raw spectral-array footprint.
cache_table = cache_table.join(
    cache_table["path"].apply(cache_partition_sizes)
)
display(
    cache_table[
        ["valid", "entry_count", "stored_mib", "raw_array_mib", "path"]
    ].round({"stored_mib": 1, "raw_array_mib": 1})
)
print(
    f"Total cache size for the active classes: "
    f"{cache_table['stored_mib'].sum():.1f} MiB for "
    f"{int(cache_table['entry_count'].sum())} neutral Warp source grids."
)
finish_stage(
    "3. Build or validate source cache",
    stage_started,
    cache_entries=int(cache_table["entry_count"].sum()),
    cache_mib=float(cache_table["stored_mib"].sum()),
)

## 4. Plan diverse survey cadences

`survey_name` may be `"ztf"`, `"lsst"`, or `"combined"`. This stage reads only inexpensive archive bounds and plans many 360-day source windows. Automatic starts are spread evenly across each instrument's full available time span. Neighboring windows may overlap, so a dense set of starts samples different local observing patterns without pretending that every realization is statistically independent. Each window is shifted onto the same simulation interval while preserving its internal cadence. The large cadence tables are deliberately not loaded until the sequential simulation loop.

In [ ]:
# Plan bounded cadence realizations without loading their large observation tables
stage_started = start_stage("4. Plan survey realizations")
survey_factory = SurveyFactory()

# Optional explicit starts override the even archive-wide placement per instrument.
custom_source_starts = {}
if custom_ztf_source_mjd_starts is not None:
    custom_source_starts["ztf"] = custom_ztf_source_mjd_starts
if custom_lsst_source_mjd_starts is not None:
    custom_source_starts["lsst"] = custom_lsst_source_mjd_starts
cadence_realizations = survey_factory.plan_relative_realizations(
    survey_name,
    tstart=observation_tstart,
    tstop=observation_tstop,
    count=survey_realization_count,
    options=survey_options,
    source_mjd_starts=custom_source_starts or None,
    require_non_overlapping=require_non_overlapping_realizations,
)

# Make the independent source windows visible before expensive simulation starts.
realization_plan_rows = []
for realization, target_count in zip(cadence_realizations, realization_sizes):
    row = {
        "realization_id": realization.realization_id,
        "target_count": target_count,
        "simulation_mjd_start": realization.requested_mjd_range[0],
        "simulation_mjd_stop": realization.requested_mjd_range[1],
    }
    for instrument, bounds in realization.source_mjd_ranges.items():
        row[f"{instrument}_source_start"] = bounds[0]
        row[f"{instrument}_source_stop"] = bounds[1]
    realization_plan_rows.append(row)
realization_plan = pd.DataFrame(realization_plan_rows)
display(realization_plan)
# Show how densely the sliding starts sample each survey archive.
for instrument in ("ztf", "lsst"):
    start_column = f"{instrument}_source_start"
    if start_column in realization_plan and len(realization_plan) > 1:
        start_spacing = np.diff(realization_plan[start_column].to_numpy())
        print(
            f"{instrument.upper()} window-start spacing: "
            f"{start_spacing.min():.1f}--{start_spacing.max():.1f} days; "
            f"window length: {duration_days:g} days."
        )
finish_stage(
    "4. Plan survey realizations",
    stage_started,
    realizations=len(cadence_realizations),
    days_per_realization=duration_days,
)

## 5. Configure the fitclass-local bounded simulation

The total target count is divided across the planned cadence realizations. Exact raw-class counts are derived from one global seven-class allocation, then sliced across the realizations so local integer rounding cannot accumulate into an SLSN bias. Every realization receives its own deterministic seed. Batches never mix fitclasses and close at the fitclass end, the target-row limit, or before a new selected source key would exceed the source-key limit. Peak dates are restricted to the protected interior of each 360-day observing window.

In [ ]:
# Define the reproducible population and initialize the bounded simulation runner
stage_started = start_stage("5. Configure bounded simulation")
# Derive distinct stable seeds so realizations never repeat the same population draw.
def realization_seed(base_seed, realization_id):
    """Derive one deterministic NumPy-compatible seed per cadence window."""

    payload = f"{base_seed}|{realization_id}".encode()
    return int.from_bytes(blake2b(payload, digest_size=8).digest(), "little")

# Each specification fixes one population shard and its exact source-window options.
sample_specs = []
for realization, target_count, class_counts in zip(
    cadence_realizations, realization_sizes, realization_class_counts
):
    sample_specs.append(
        WarpSampleSpec(
            run_name=f"{run_name}__{realization.realization_id}",
            active_fitclasses=active_fitclasses,
            size=target_count,
            class_sampling="counts",
            class_counts=class_counts,
            redshift_sampling="uniform",
            zmax=zmax,
            tstart=peak_tstart,
            tstop=peak_tstop,
            color_mode="draw",
            survey_name=survey_name,
            survey_options=realization.survey_options,
            survey_realization_id=realization.realization_id,
            batch_size=min(target_count, batch_size),
            max_sources_per_batch=max_sources_per_batch,
            phase_range=simulation_phase_range,
            incl_error=include_noise,
            seed=realization_seed(seed, realization.realization_id),
        )
    )
# The runner consumes cached neutral sources and applies event-specific parameters lazily.
runner = WarpSimulationRunner(
    warpcoeffs_dir,
    source_cache_dir=source_cache_dir,
)
display(
    pd.DataFrame(
        {
            "realization_id": [item.realization_id for item in cadence_realizations],
            "run_name": [spec.run_name for spec in sample_specs],
            "targets": [spec.size for spec in sample_specs],
            "seed": [spec.seed for spec in sample_specs],
            "batch_size": [spec.batch_size for spec in sample_specs],
        }
    )
)
finish_stage(
    "5. Configure bounded simulation",
    stage_started,
    targets=sum(spec.size for spec in sample_specs),
    realizations=len(sample_specs),
    batch_size=batch_size,
)

## 6. Run and persist the simulation

The realization loop loads one 360-day cadence, simulates only its target shard, persists it, and releases the cadence before loading the next. Thus peak RAM follows the largest single window instead of the sum of all windows. With `keep_warpcoeffs_in_memory=True`, the small immutable coefficient dictionaries are loaded on the first realization, reused by the remaining realizations, and cleared once at the end. The runner persists truth directly from each complete target batch. Completed Parquet pairs are resumable; an interrupted batch is recomputed from its beginning. A top-level ensemble manifest records every source window and child run. `progress_every_batches` controls compact live reports within each realization.

In [ ]:
# Load, simulate, and release every cadence realization sequentially
stage_started = start_stage("6. Simulate and persist batches")
ensemble_dir = output_dir / run_name
ensemble_dir.mkdir(parents=True, exist_ok=True)
manifests = []
realization_results = []
completed_batches = []
survey_summaries = []

# Process one large survey DataFrame at a time to cap live cadence memory.
for realization, sample_spec in zip(cadence_realizations, sample_specs):
    realization_started = time.perf_counter()
    print(
        f"[Realization {realization.index + 1}/{realization.count}] "
        f"{realization.realization_id}: loading {duration_days:g}-day cadence for "
        f"{sample_spec.size:,} targets"
    )
    cadence_load_started = time.perf_counter()
    survey = survey_factory.create(
        survey_name,
        tstart=observation_tstart,
        tstop=observation_tstop,
        options=realization.survey_options,
    )
    cadence_load_seconds = time.perf_counter() - cadence_load_started
    footprint_geometry = survey.warp_provenance["footprint_geometry"]
    if footprint_geometry["field_geometry_method"] != survey_geometry_contract:
        raise RuntimeError("The survey does not use the expected geometry contract.")

    # Persisted child manifests retain exact cadence and source-file provenance.
    simulation_started = time.perf_counter()
    manifest = runner.run(
        sample_spec,
        ensemble_dir,
        survey=survey,
        resume=True,
        progress_every_batches=progress_every_batches,
        retain_coefficient_cache=keep_warpcoeffs_in_memory,
    )
    simulation_seconds = time.perf_counter() - simulation_started
    print(
        f"[Realization done] {realization.realization_id}: "
        f"cadence load {cadence_load_seconds:.1f} s | "
        f"simulation/persist {simulation_seconds:.1f} s"
    )
    if manifest["schema_version"] != SIMULATION_SCHEMA_VERSION:
        raise RuntimeError("A child run produced an incompatible schema version.")
    manifests.append(manifest)
    child_run_dir = ensemble_dir / sample_spec.run_name
    realization_results.append(
        {
            "realization_id": realization.realization_id,
            "run_name": sample_spec.run_name,
            "run_directory": str(child_run_dir),
            "truth_rows": int(manifest["truth_rows"]),
            "observation_rows": int(manifest["observation_rows"]),
            "batches": len(manifest["batches"]),
            "cadence_rows": len(survey.data),
            "geometric_area_deg2": survey.simulation_skyarea.area_deg2,
            "cadence_load_seconds": cadence_load_seconds,
            "simulation_seconds": simulation_seconds,
            "total_seconds": time.perf_counter() - realization_started,
        }
    )
    survey_summaries.append(
        {
            "realization_id": realization.realization_id,
            "cadence_rows": len(survey.data),
            "bands": ", ".join(sorted(survey.data["band"].unique())),
            "mjd_min": float(survey.data["mjd"].min()),
            "mjd_max": float(survey.data["mjd"].max()),
            "area_deg2": survey.simulation_skyarea.area_deg2,
            "observed_field_counts": footprint_geometry["observed_field_counts"],
        }
    )
    for batch_id, batch in manifest["batches"].items():
        if batch["status"] == "complete":
            completed_batches.append(
                {
                    **batch,
                    "batch_id": batch_id,
                    "realization_id": realization.realization_id,
                    "run_dir": child_run_dir,
                }
            )

    # Remove the current cadence before constructing the next multi-GB DataFrame.
    del survey
    gc.collect()

# Coefficients are tiny compared with a cadence table; release them once, at the end.
if keep_warpcoeffs_in_memory:
    runner.clear_coefficient_cache()

# Write a compact top-level index for reproducible downstream dataset splitting.
ensemble_manifest = {
    "ensemble_schema_version": 1,
    "status": "complete",
    "configuration_digest": configuration_digest,
    "configuration": configuration,
    "truth_rows": sum(item["truth_rows"] for item in realization_results),
    "observation_rows": sum(
        item["observation_rows"] for item in realization_results
    ),
    "windows_may_overlap": not require_non_overlapping_realizations,
    "realizations": [
        {
            **realization.to_dict(),
            **result,
        }
        for realization, result in zip(cadence_realizations, realization_results)
    ],
    "split_policy": (
        "Complete realization_id values may be assigned to separate splits."
        if require_non_overlapping_realizations
        else (
            "Sliding windows share cadence rows. Define disjoint source-time "
            "blocks before assigning train, validation, and test data; grouping "
            "only by realization_id does not prevent cadence leakage."
        )
    ),
}
ensemble_manifest_path = ensemble_dir / "ensemble_manifest.json"
temporary_manifest_path = ensemble_dir / "ensemble_manifest.json.tmp"
temporary_manifest_path.write_text(
    json.dumps(ensemble_manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)
os.replace(temporary_manifest_path, ensemble_manifest_path)
display(pd.DataFrame(realization_results))
display(pd.DataFrame(survey_summaries))
finish_stage(
    "6. Simulate and persist batches",
    stage_started,
    targets=ensemble_manifest["truth_rows"],
    observations=ensemble_manifest["observation_rows"],
    batches=len(completed_batches),
    realizations=len(manifests),
)

## 7. Select one comparable example from every CC class

For large simulations, at most `analysis_objects_per_fitclass` persisted targets per class are retained for plotting, divided as evenly as possible across all cadence realizations. Individual Parquet batches are filtered while they are read, so 100 small realization quotas do not inflate into 100 complete batches per class. Set the limit to `None` only when the full result safely fits in RAM. Within this bounded analysis subset, selection keeps objects with at least one significant detection but strictly fewer than 100, then chooses the largest remaining detection count. Proximity to the comparison redshift and the object ID are used only as deterministic tie-breakers.

In [ ]:
# Read a realization-balanced subset of persisted partitions without rebuilding models
stage_started = start_stage("7. Load analysis subset and select examples")
# Allocate an exact per-class target budget across all cadence realizations.
if analysis_objects_per_fitclass is None:
    realization_quotas = None
else:
    quota_base, quota_remainder = divmod(
        analysis_objects_per_fitclass, survey_realization_count
    )
    realization_quotas = {
        realization.realization_id: quota_base + (index < quota_remainder)
        for index, realization in enumerate(cadence_realizations)
    }

# Select rows inside each truth batch rather than retaining an entire batch per window.
selected_rows_by_key = {}
analysis_batch_selections = []
truth_frames = []
for batch in completed_batches:
    fitclass = str(batch["fitclass"])
    realization_id = str(batch["realization_id"])
    key = (realization_id, fitclass)
    if realization_quotas is None:
        remaining = int(batch["truth_rows"])
    else:
        remaining = (
            realization_quotas[realization_id]
            - selected_rows_by_key.get(key, 0)
        )
    if remaining <= 0:
        continue
    truth_frame = pd.read_parquet(batch["run_dir"] / batch["truth"])
    truth_frame = truth_frame.head(remaining).copy()
    if truth_frame.empty:
        continue
    truth_frame["survey_realization_id"] = realization_id
    selected_object_ids = set(truth_frame["object_id"].astype(str))
    truth_frames.append(truth_frame)
    analysis_batch_selections.append((batch, selected_object_ids))
    selected_rows_by_key[key] = (
        selected_rows_by_key.get(key, 0) + len(truth_frame)
    )

truth = pd.concat(truth_frames, ignore_index=True)
analysis_batches = [batch for batch, _ in analysis_batch_selections]
analysis_truth_rows = len(truth)
print(
    f"Analysis subset: {analysis_truth_rows:,}/"
    f"{int(ensemble_manifest['truth_rows']):,} targets from "
    f"{len(analysis_batches):,}/{len(completed_batches):,} batches."
)

# Read observation partitions one at a time and keep only selected objects.
observation_frames = []
for batch, selected_object_ids in analysis_batch_selections:
    observation_frame = pd.read_parquet(
        batch["run_dir"] / batch["observations"]
    )
    observation_frame = observation_frame.loc[
        observation_frame["object_id"].astype(str).isin(selected_object_ids)
    ].copy()
    if observation_frame.empty:
        continue
    observation_frame["survey_realization_id"] = batch["realization_id"]
    observation_frames.append(observation_frame)
if not observation_frames:
    raise RuntimeError("The bounded analysis subset contains no observations.")
observations = pd.concat(observation_frames, ignore_index=True)

# Attach peak times and derive observer-frame phase, instrument, and S/N per row.
observations = observations.loc[observations["object_id"].notna()].copy()
observations = observations.merge(
    truth[["object_id", "t0"]],
    on="object_id",
    how="left",
    validate="many_to_one",
)
observations["phase_observer"] = observations["mjd"] - observations["t0"]
observations["instrument"] = observations["band"].str.extract(
    r"^(ztf|lsst)"
)[0].fillna("other").str.upper()
observations["snr"] = (
    observations["flux"] / observations["fluxerr"]
)
observations["is_magnitude_detection"] = (
    np.isfinite(observations["snr"])
    & (observations["flux"] > 0.0)
    & (observations["snr"] >= minimum_plot_snr)
)

if observations.empty:
    raise RuntimeError(
        "The simulation produced no observations. Increase the time window "
        "or check the survey and phase-range settings."
    )

# Summarize cadence and significant detections for every simulated object.
observation_stats = observations.groupby("object_id").agg(
    n_observations=("mjd", "size"),
    n_bands=("band", "nunique"),
    n_instruments=("instrument", "nunique"),
    n_magnitude_detections=("is_magnitude_detection", "sum"),
    median_snr=(
        "flux",
        lambda values: float(
            np.nanmedian(
                np.abs(values.to_numpy())
                / observations.loc[values.index, "fluxerr"].to_numpy()
            )
        ),
    ),
)
detection_stats = (
    observations.loc[observations["is_magnitude_detection"]]
    .groupby("object_id")
    .agg(
        n_detected_bands=("band", "nunique"),
        n_detecting_instruments=("instrument", "nunique"),
        first_detected_phase=("phase_observer", "min"),
        last_detected_phase=("phase_observer", "max"),
        pre_peak_detections=(
            "phase_observer",
            lambda values: int((values < 0.0).sum()),
        ),
        post_peak_detections=(
            "phase_observer",
            lambda values: int((values >= 0.0).sum()),
        ),
    )
)
observation_stats = observation_stats.join(detection_stats).fillna(
    {
        "n_detected_bands": 0,
        "n_detecting_instruments": 0,
        "pre_peak_detections": 0,
        "post_peak_detections": 0,
    }
)
observation_stats["detected_phase_span"] = (
    observation_stats["last_detected_phase"]
    - observation_stats["first_detected_phase"]
)
observation_stats["has_pre_and_post_peak"] = (
    (observation_stats["pre_peak_detections"] > 0)
    & (observation_stats["post_peak_detections"] > 0)
)

# Combine physical truth with observational statistics before choosing examples.
selection_table = truth.merge(
    observation_stats,
    left_on="object_id",
    right_index=True,
    how="inner",
)
selection_table["redshift_distance"] = np.abs(
    selection_table["z"] - comparison_redshift
)
# Exclude zero-detection objects and rare high-cadence cases at or above the ceiling.
selection_table = selection_table.loc[
    (selection_table["n_magnitude_detections"] > 0)
    & (
        selection_table["n_magnitude_detections"]
        < maximum_example_detections
    )
].copy()
# Select the largest allowed detection count per class; remaining keys only break ties.
selection_table = selection_table.sort_values(
    [
        "fitclass",
        "n_magnitude_detections",
        "redshift_distance",
        "object_id",
    ],
    ascending=[True, False, True, True],
    kind="stable",
)
selected_examples = (
    selection_table.groupby("fitclass", sort=False, as_index=False)
    .head(1)
    .copy()
)
selected_examples["fitclass"] = pd.Categorical(
    selected_examples["fitclass"],
    categories=active_fitclasses,
    ordered=True,
)
selected_examples = selected_examples.sort_values("fitclass")
missing_examples = set(active_fitclasses) - set(
    selected_examples["fitclass"].astype(str)
)
if missing_examples:
    raise RuntimeError(
        f"No example with 0 < detections < {maximum_example_detections} "
        "was available for: "
        + ", ".join(sorted(missing_examples))
    )

# Restrict the full observation table to the selected object from each fitclass.
selected_object_ids = selected_examples["object_id"].astype(str).tolist()
selected_observations = observations.loc[
    observations["object_id"].isin(selected_object_ids)
].copy()
selected_observations = selected_observations.merge(
    selected_examples[
        [
            "object_id",
            "fitclass",
            "basis_sn",
            "template_sn",
            "z",
            "survey_realization_id",
        ]
    ],
    on="object_id",
    how="left",
    suffixes=("", "_truth"),
)
magnitude_lightcurves = selected_observations.loc[
    selected_observations["is_magnitude_detection"]
].copy()
if magnitude_lightcurves.empty:
    raise RuntimeError(
        f"No selected class example has a measurement with S/N >= "
        f"{minimum_plot_snr:g}. Lower minimum_plot_snr in USER SETTINGS "
        "or increase sample_size."
    )
# Convert only positive significant fluxes to AB magnitudes and propagated errors.
magnitude_lightcurves["magnitude"] = (
    magnitude_lightcurves["zp"]
    - 2.5 * np.log10(magnitude_lightcurves["flux"])
)
magnitude_lightcurves["magnitude_error"] = (
    2.5 / np.log(10.0)
    * magnitude_lightcurves["fluxerr"]
    / magnitude_lightcurves["flux"]
)

display(
    selected_examples[
        [
            "survey_realization_id",
            "fitclass",
            "basis_sn",
            "template_sn",
            "z",
            "distance_modulus",
            "magabs",
            "template_prob",
            "entry_sampling_probability",
            "n_detecting_instruments",
            "n_detected_bands",
            "first_detected_phase",
            "last_detected_phase",
            "detected_phase_span",
            "n_magnitude_detections",
            "median_snr",
        ]
    ].reset_index(drop=True)
)
finish_stage(
    "7. Load analysis subset and select examples",
    stage_started,
    loaded_targets=len(truth),
    loaded_observations=len(observations),
    loaded_batches=len(analysis_batches),
)

## 8. Compare the selected magnitude light curves

Each panel identifies the Warp base supernova and the original `sncosmo` source. LSST measurements use larger star markers and are drawn above the smaller circular ZTF points so the two instruments remain easy to distinguish. The cache filename is intentionally omitted because it is an implementation detail shared by many simulated objects.

In [ ]:
# Plot one magnitude light curve per class with explicit base and source provenance
stage_started = start_stage("8. Plot selected light curves")
# Use a shared color per physical band while marker shape identifies the instrument.
band_colors = {
    "ztfg": "tab:green",
    "ztfr": "tab:red",
    "ztfi": "tab:orange",
    "lsstu": "tab:purple",
    "lsstg": "limegreen",
    "lsstr": "crimson",
    "lssti": "darkorange",
    "lsstz": "saddlebrown",
    "lssty": "black",
}
# Larger star markers and a higher z-order make LSST stand out from ZTF circles.
instrument_styles = {
    "ZTF": {
        "marker": "o",
        "markersize": 4.0,
        "alpha": 0.75,
        "zorder": 2,
        "markeredgewidth": 0.5,
    },
    "LSST": {
        "marker": "*",
        "markersize": 8.0,
        "alpha": 0.95,
        "zorder": 3,
        "markeredgewidth": 0.7,
    },
    "OTHER": {
        "marker": "^",
        "markersize": 4.5,
        "alpha": 0.8,
        "zorder": 2,
        "markeredgewidth": 0.5,
    },
}

# Derive a compact two-column panel grid from the active class count.
n_classes = len(active_fitclasses)
n_columns = 2
n_rows = int(np.ceil(n_classes / n_columns))
fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(15, 4.1 * n_rows),
    sharex=True,
    sharey=True,
    squeeze=False,
)

# Plot each instrument-band group separately so its legend entry and style stay explicit.
legend_handles = {}
for ax, fitclass in zip(axes.flat, active_fitclasses):
    example = selected_examples.loc[
        selected_examples["fitclass"].astype(str).eq(fitclass)
    ].iloc[0]
    class_lightcurve = magnitude_lightcurves.loc[
        magnitude_lightcurves["object_id"].eq(example["object_id"])
    ]
    for (instrument, band), band_data in class_lightcurve.groupby(
        ["instrument", "band"],
        sort=True,
    ):
        style = instrument_styles.get(instrument, instrument_styles["OTHER"])
        handle = ax.errorbar(
            band_data["phase_observer"],
            band_data["magnitude"],
            yerr=band_data["magnitude_error"],
            fmt=style["marker"],
            color=band_colors.get(band, "0.4"),
            ms=style["markersize"],
            markeredgewidth=style["markeredgewidth"],
            capsize=1.2,
            alpha=style["alpha"],
            zorder=style["zorder"],
            linestyle="none",
            label=f"{instrument} {band}",
        )
        legend_handles.setdefault(f"{instrument} {band}", handle)
    ax.axvline(0.0, color="0.25", linestyle="--", linewidth=0.9)
    ax.set_title(
        f"{fitclass} | cadence={example['survey_realization_id']} | "
        f"z={float(example['z']):.4f} | "
        f"M={float(example['magabs']):.2f} | "
        f"N_det={int(example['n_magnitude_detections'])}\n"
        f"Base SN: {example['basis_sn']}\n"
        f"sncosmo source: {example['template_sn']}",
        fontsize=10,
    )
    ax.grid(alpha=0.2)

# Apply shared labels and astronomical magnitude orientation after filling all panels.
for ax in axes[-1, :]:
    ax.set_xlabel("Observer-frame phase relative to t0 [days]")
for ax in axes[:, 0]:
    ax.set_ylabel("AB magnitude")
for ax in axes.flat[n_classes:]:
    ax.set_visible(False)
axes.flat[0].invert_yaxis()
# Build one figure-level legend from the first occurrence of each instrument-band pair.
fig.legend(
    legend_handles.values(),
    legend_handles.keys(),
    loc="upper center",
    ncol=min(5, len(legend_handles)),
    fontsize="small",
    bbox_to_anchor=(0.5, 1.0),
)
fig.suptitle(
    f"Largest CC-SN detection count below {maximum_example_detections} "
    f"per fitclass; S/N >= {minimum_plot_snr:g}",
    y=1.025,
)
fig.tight_layout()
#display(fig)
finish_stage("8. Plot selected light curves", stage_started, panels=n_classes)

## 9. Inspect the comparison provenance

The summary and measurement preview make the class, base supernova, `sncosmo` source, redshift, distance modulus, raw `template_prob`, effective `entry_sampling_probability`, absolute magnitude, phase coverage, and retained detections directly traceable. The final tables compare notebook-stage time and peak RAM, aggregate per-batch timing distributions, and project storage for one million targets from the current run.

In [ ]:
# Inspect the selected class examples and their Warp provenance
stage_started = start_stage("9. Inspect provenance")
# Keep selection inputs and provenance together for a compact per-class audit table.
comparison_summary_columns = [
    "survey_realization_id",
    "fitclass",
    "object_id",
    "basis_sn",
    "template_sn",
    "z",
    "t0",
    "distance_modulus",
    "magabs",
    "template_prob",
    "entry_sampling_probability",
    "n_detecting_instruments",
    "n_detected_bands",
    "first_detected_phase",
    "last_detected_phase",
    "detected_phase_span",
    "pre_peak_detections",
    "post_peak_detections",
    "n_magnitude_detections",
    "n_observations",
]
comparison_summary = selected_examples[comparison_summary_columns].copy()
comparison_summary["plotted_measurements"] = comparison_summary["object_id"].map(
    magnitude_lightcurves.groupby("object_id").size()
).fillna(0).astype(int)
comparison_summary["excluded_measurements"] = (
    comparison_summary["n_observations"]
    - comparison_summary["plotted_measurements"]
)
display(comparison_summary.reset_index(drop=True))

# Preview a few chronological plotted rows per class without flooding the notebook.
inspection_columns = [
    "survey_realization_id",
    "fitclass",
    "basis_sn",
    "template_sn",
    "mjd",
    "phase_observer",
    "instrument",
    "band",
    "magnitude",
    "magnitude_error",
    "snr",
]
display(
    magnitude_lightcurves[inspection_columns]
    .sort_values(["fitclass", "mjd"])
    .groupby("fitclass", observed=True, group_keys=False)
    .head(5)
    .reset_index(drop=True)
)
print(
    f"Magnitude plots retain {len(magnitude_lightcurves):,} positive measurements with "
    f"S/N >= {minimum_plot_snr:g} across {len(selected_examples)} CC-SN classes."
)
finish_stage(
    "9. Inspect provenance",
    stage_started,
    preview_rows=min(len(magnitude_lightcurves), 5 * len(selected_examples)),
)

# Compare notebook-stage wall time and peak kernel memory at a glance.
stage_profile = pd.DataFrame(stage_metrics)
stage_profile["seconds"] = stage_profile["seconds"].round(2)
stage_profile["peak_rss_gib"] = stage_profile["peak_rss_gib"].round(2)
stage_profile["peak_increase_gib"] = (
    stage_profile["peak_increase_gib"].round(2)
)
display(stage_profile)
slowest_stage = stage_profile.set_index("stage")["seconds"].idxmax()
largest_memory_stage = stage_profile.set_index("stage")["peak_increase_gib"].idxmax()
print(f"Slowest notebook stage: {slowest_stage}")
print(f"Largest peak-RAM increase: {largest_memory_stage}")
stage_tuning_hints = {
    "3. Build or validate source cache": (
        "Reuse the cache and place custom_source_cache_dir on fast local storage."
    ),
    "4. Plan survey realizations": (
        "Use fewer realizations or provide explicit valid source-window starts."
    ),
    "6. Simulate and persist batches": (
        "Use the per-batch table below to distinguish SkySurvey from Parquet I/O."
    ),
    "7. Load analysis subset and select examples": (
        "Lower analysis_objects_per_fitclass; persisted simulation data is unaffected."
    ),
    "8. Plot selected light curves": (
        "Reduce active_fitclasses or plotted bands for exploratory runs."
    ),
}
if slowest_stage in stage_tuning_hints:
    print(stage_tuning_hints[slowest_stage])

# Aggregate persisted per-batch timings without double-counting nested model work.
batch_profile = pd.DataFrame(
    [
        {
            "realization_id": batch["realization_id"],
            "batch_id": batch["batch_id"],
            "fitclass": batch["fitclass"],
            "truth_rows": int(batch["truth_rows"]),
            "observation_rows": int(batch["observation_rows"]),
            **batch["timing_seconds"],
        }
        for batch in completed_batches
    ]
)
timing_columns = [
    "target_drawing",
    "source_cache_loading",
    "model_building",
    "skysurvey_total",
    "parquet_writing",
]
timing_summary = pd.DataFrame(
    {
        "total_seconds": batch_profile[timing_columns].sum(),
        "median_batch_seconds": batch_profile[timing_columns].median(),
        "p95_batch_seconds": batch_profile[timing_columns].quantile(0.95),
        "max_batch_seconds": batch_profile[timing_columns].max(),
    }
).round(3)
display(timing_summary)
# Compare cadence loading and simulation throughput across survey realizations.
realization_profile = pd.DataFrame(realization_results).set_index(
    "realization_id"
)
realization_profile["targets_per_simulation_second"] = (
    realization_profile["truth_rows"]
    / realization_profile["simulation_seconds"]
)
display(
    realization_profile[
        [
            "truth_rows",
            "observation_rows",
            "cadence_rows",
            "cadence_load_seconds",
            "simulation_seconds",
            "targets_per_simulation_second",
        ]
    ].round(2)
)

# Estimate storage scaling from the files produced by this exact run.
truth_bytes = sum(
    (batch["run_dir"] / batch["truth"]).stat().st_size
    for batch in completed_batches
)
observation_bytes = sum(
    (batch["run_dir"] / batch["observations"]).stat().st_size
    for batch in completed_batches
)
bytes_per_target = (
    (truth_bytes + observation_bytes) / ensemble_manifest["truth_rows"]
)
exclusive_batch_seconds = (
    batch_profile["target_drawing"].sum()
    + batch_profile["skysurvey_total"].sum()
    + batch_profile["parquet_writing"].sum()
)
batch_seconds_per_target = (
    exclusive_batch_seconds / ensemble_manifest["truth_rows"]
)
mean_targets_per_batch = ensemble_manifest["truth_rows"] / len(batch_profile)
storage_projection = pd.Series(
    {
        "current_run_gib": (truth_bytes + observation_bytes) / 1024**3,
        "bytes_per_target": bytes_per_target,
        "projected_gib_for_1m_targets": bytes_per_target * 1_000_000 / 1024**3,
        "batch_seconds_per_target": batch_seconds_per_target,
        "projected_compute_hours_for_1m_targets": (
            batch_seconds_per_target * 1_000_000 / 3600.0
        ),
        "projected_batches_for_1m_targets": (
            1_000_000 / mean_targets_per_batch
        ),
    },
    name="value",
)
display(storage_projection.round(3))

# Report the dominant exclusive phase to guide the next stress-test adjustment.
exclusive_totals = timing_summary.loc[
    ["target_drawing", "skysurvey_total", "parquet_writing"],
    "total_seconds",
]
dominant_phase = exclusive_totals.idxmax()
tuning_hints = {
    "target_drawing": "Population drawing dominates: profile sampling and coefficient loading.",
    "skysurvey_total": (
        "SkySurvey dominates: tune batch_size, filters, duration_days, and "
        "simulation_phase_range while watching peak RSS."
    ),
    "parquet_writing": (
        "Parquet output dominates: use larger safe batches or faster output storage."
    ),
}
print(f"Dominant exclusive phase: {dominant_phase}")
print(tuning_hints[dominant_phase])